# Prepare agent trajectories for SFT

Normalize every recorded attempt, inspect validation issues, and write JSONL. No training is run. Sensitive-data detection is heuristic; review outputs before use.

In [1]:
import logging
import sys
from pathlib import Path
from datasets import load_dataset

logging.basicConfig(level=logging.INFO)
project = Path.cwd()
while not (project / "model_training").is_dir() and project.parent != project:
    project = project.parent
lab = project / "model_training/post_training/07-agent_training/agent_trace_grpo_lab"
sys.path.insert(0, str(lab))
from interfaces import SWEGymTraceAdapter, normalize_trace, write_sft_dataset
print("[STAGE] Setup")
print("✓ Lab:", lab)

[STAGE] Setup
✓ Lab: /Users/vedaangchopra/all_data/complete_technical_work/reference_and_learning_content/deep_learning/model_training/post_training/07-agent_training/agent_trace_grpo_lab


In [2]:
dataset_name = "SWE-Gym/OpenHands-Sampled-Trajectories"
split = "train.raw"
print("[STAGE] Load", dataset_name, split)
raw = load_dataset(dataset_name, split=split)
print("✓ Trajectories:", len(raw))

[STAGE] Load SWE-Gym/OpenHands-Sampled-Trajectories train.raw
✓ Trajectories: 6055


In [3]:
print("[STAGE] Normalize one attempt")
adapter = SWEGymTraceAdapter()
sample, sample_issues = normalize_trace(adapter.parse(raw[0]))
print("✓ Messages:", len(sample["messages"]), "issues:", len(sample_issues))
print("Roles:", [m["role"] for m in sample["messages"][:10]])
print("Issue types:", sorted({i["type"] for i in sample_issues}))

[STAGE] Normalize one attempt
✓ Messages: 13 issues: 2
Roles: ['system', 'user', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'tool']
Issue types: ['incomplete_trajectory', 'unmatched_tool_call']


In [4]:
print("[STAGE] Check empty trace")
empty, empty_issues = normalize_trace(adapter.parse({"messages": []}))
assert not empty["messages"]
assert any(i["type"] == "incomplete_trajectory" for i in empty_issues)
assert len(sample["messages"]) == len(raw[0]["messages"])
print("✓ Empty trajectory is flagged; sample message order is retained")

[STAGE] Check empty trace
✓ Empty trajectory is flagged; sample message order is retained


<function sft_data.normalize_trace(trace: Any) -> tuple[dict[str, typing.Any], list[dict[str, typing.Any]]]>

In [ ]:
from datetime import datetime
output_dir = project / "execution_results/agent_trace_sft" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
print("[STAGE] Write normalized data and validation")
summary = write_sft_dataset(raw, adapter, output_dir / "trajectories.jsonl", output_dir / "summary.json", output_dir / "issues.jsonl")
assert summary["total_trajectories"] == len(raw)
print("✓", summary)
print("Artifacts:", output_dir)

INFO:sft_data:Preparing SFT records at /Users/vedaangchopra/all_data/complete_technical_work/reference_and_learning_content/deep_learning/execution_results/agent_trace_sft/trajectories.jsonl


[STAGE] Write normalized data and validation


## Review

`issues.jsonl` maps validation issues to zero-based source row indices. All attempts remain in the output; invalid or incomplete rows need review before training. Redaction is heuristic and cannot guarantee removal of every sensitive value. Tool use also depends on the chosen model chat template and dataset library version.